<a href="https://colab.research.google.com/github/prithwis/Centaur/blob/main/MOS_WorldFactCollector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![alt text](https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/images/CentaurBanner2.png)<br>


<hr>

[Prithwis Mukerjee](http://www.linkedin.com/in/prithwis)<br>

In [10]:
!pip -q install pymupdf
!wget -O mos.py -q https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/mosUtils/mos.py                  # all
!wget -O mos1.py -q https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/mosUtils/mos1.py                  # all
import mos, mos1
from pathlib import Path
import time

In [2]:
import importlib
#importlib.reload(mos1)

In [ ]:
client = mos.authenticateOpenAI()

OpenAI authentication successful ✔
Logged in as Calcutta prithwis@yantrajaal.com


In [3]:
from google.colab import drive
drive.mount('/content/drive')

WORLDFACTS = "/content/drive/MyDrive/Praxis/Course Geopolitics/Centaur Project/WorldFacts"

!ls -l "{WORLDFACTS}"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
total 8
drwx------ 2 root root 4096 Sep 28 09:09 MD
drwx------ 2 root root 4096 Sep 28 06:55 PDF


## Static World Data
URLs stored as PDF files that are converted to markdown files

In [ ]:
# =============================================================================
# MOS STATIC SOURCE PACK v1
# =============================================================================

# 01. IEA — Oil Market Report, September 2026
# Global oil supply, demand, inventories, prices, refining and Gulf disruption.
# https://www.iea.org/reports/oil-market-report-september-2026

# 02. US EIA — Short-Term Energy Outlook: Global Oil Markets
# Oil prices, Middle East production/exports, Iran and Hormuz disruption.
# https://www.eia.gov/outlooks/steo/report/global_oil.php

# 03. PPAC — India's Oil & Gas Ready Reckoner / Current Reports
# India's crude production, imports, consumption, refining and import dependence.
# https://ppac.gov.in/
#Snapshot of India's Oil and Gas Data → Monthly Ready Reckoner — August 2026 → View Report (PDF)

# 04. Government of India / PIB — West Asia Energy Supply Disruption
# India's Gulf/Hormuz exposure and government energy-security response.
# https://www.pib.gov.in/PressReleasePage.aspx?PRID=2239021&lang=1&reg=6


# 06. World Bank — Iran Macro Poverty Outlook, April 2026
# Concise current picture of Iran's economy, sanctions, inflation, fiscal stress and oil vulnerability.
# https://documents.worldbank.org/en/publication/documents-reports/documentdetail/099211508192619381
# https://documents1.worldbank.org/curated/en/099211508192619381/pdf/Bulk-8dc686b3-acf3-4bd1-92b1-373dd5db8389.pdf

# 07. SIPRI — Yearbook 2026 Summary
# Regional strategic context, military capabilities and nuclear forces.
# https://www.sipri.org/yearbook/2026/summary/sipri-yearbook-2026-summary-english

# 08. CSIS Missile Threat — Iran
# Iranian ballistic/cruise missiles, anti-ship systems and UAV capabilities.
# https://missilethreat.csis.org/country/iran/

# =============================================================================

In [ ]:
PDF_DIR = f"{WORLDFACTS}/PDF"
MD_DIR  = f"{WORLDFACTS}/MD"

!ls -l "{PDF_DIR}"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
total 19681
-rw------- 1 root root 10479230 Sep 28 21:29  1790418308_Final_Snapshot_of_Indias_Oil_Gas_data_augpages.pdf
-rw------- 1 root root   153646 Sep 28 21:39  Iran_Data.pdf
-rw------- 1 root root  7057875 Sep 28 21:41 'Missiles of Iran _ Missile Threat.pdf'
-rw------- 1 root root   214516 Sep 28 03:04 'Oil Market Report - September 2026 – Analysis - IEA.pdf'
-rw------- 1 root root   259068 Sep 28 21:32 'Press Release Page _ Press Information Bureau.pdf'
-rw------- 1 root root   381243 Sep 28 21:28 'Short-Term Energy Outlook - U.S. Energy Information Administration (EIA).pdf'
-rw------- 1 root root  1605684 Sep 28 21:40  SIPRI_Summary.pdf


In [ ]:
Role_PDFReader = """
    You are a document-cleaning engine.

    Your task is to convert raw text extracted from a PDF into
    clean, well-structured Markdown.

    RULES

    1. Preserve ALL substantive factual content.
    2. Preserve all numbers, dates, quantities, percentages,
       names, units and factual claims.
    3. Preserve meaningful headings and section structure.
    4. Preserve useful tables. Reconstruct them as Markdown
       tables where reasonably possible.
    5. Remove advertisements, navigation menus, subscription
       prompts, cookie notices, social-media links, repeated
       headers and footers, page furniture and similar junk.
    6. Repair obvious line-break, hyphenation and formatting
       damage introduced by PDF extraction.
    7. Do NOT summarize.
    8. Do NOT interpret.
    9. Do NOT add information.
    10. Do NOT use outside knowledge.
    11. Do NOT correct or reconcile factual claims.
    12. If something is genuinely unclear or corrupted,
        preserve it rather than inventing content.
    13. If date of publication of document is known, state this at the top as date of publication.
        If date of publicatino of document is not clear, state date of publication not known. Do not guess

    Return ONLY the cleaned Markdown document.
    """

In [ ]:
from pathlib import Path

pdf_files = sorted(Path(PDF_DIR).glob("*.pdf"))

# ------------------------------------------------------------------
# Delete old Markdown files
# ------------------------------------------------------------------

old_md_files = list(Path(MD_DIR).glob("*.md"))

for md_file in old_md_files:
    md_file.unlink()

print(f"Deleted {len(old_md_files)} old MD files")
print(f"Found {len(pdf_files)} PDF files\n")

pdf_files = sorted(Path(PDF_DIR).glob("*.pdf"))
print(f"Found {len(pdf_files)} PDF files\n")

md_files = []
for source_PDF in pdf_files:
    print("=" * 70)
    print(f"Processing: {source_PDF.name}")
    try:
        md_path = mos1.GetWorldFile(client,Role_PDFReader,source_PDF,MD_DIR, _model="gpt-5-mini" )
        md_files.append(md_path)
    except Exception as e:
        print(f"❌ FAILED: {source_PDF.name}")
        print(f"Reason: {e}")

print("\n" + "=" * 70)
print(f"Completed: {len(md_files)} of {len(pdf_files)} files")

Deleted 7 old MD files
Found 7 PDF files

Found 7 PDF files

Processing: 1790418308_Final_Snapshot_of_Indias_Oil_Gas_data_augpages.pdf
Reading  : 1790418308_Final_Snapshot_of_Indias_Oil_Gas_data_augpages.pdf
Extracted: 46,550 characters
Created  : 1790418308FinalSnaps_260929.md
Markdown : 38,777 characters
Processing: Iran_Data.pdf
Reading  : Iran_Data.pdf
Extracted: 5,504 characters
Created  : IranData_260929.md
Markdown : 7,588 characters
Processing: Missiles of Iran _ Missile Threat.pdf
Reading  : Missiles of Iran _ Missile Threat.pdf
Extracted: 5,594 characters
Created  : MissilesofIranMissil_260929.md
Markdown : 5,672 characters
Processing: Oil Market Report - September 2026 – Analysis - IEA.pdf
Reading  : Oil Market Report - September 2026 – Analysis - IEA.pdf
Extracted: 7,853 characters
Created  : OilMarketReportSepte_260929.md
Markdown : 8,551 characters
Processing: Press Release Page _ Press Information Bureau.pdf
Reading  : Press Release Page _ Press Information Bureau.pdf
Ex

In [ ]:
print(md_files)

[PosixPath('/content/drive/MyDrive/Praxis/Course Geopolitics/Centaur Project/WorldState/MD/1790418308FinalSnaps_260929.md'), PosixPath('/content/drive/MyDrive/Praxis/Course Geopolitics/Centaur Project/WorldState/MD/IranData_260929.md'), PosixPath('/content/drive/MyDrive/Praxis/Course Geopolitics/Centaur Project/WorldState/MD/MissilesofIranMissil_260929.md'), PosixPath('/content/drive/MyDrive/Praxis/Course Geopolitics/Centaur Project/WorldState/MD/OilMarketReportSepte_260929.md'), PosixPath('/content/drive/MyDrive/Praxis/Course Geopolitics/Centaur Project/WorldState/MD/PressReleasePagePres_260929.md'), PosixPath('/content/drive/MyDrive/Praxis/Course Geopolitics/Centaur Project/WorldState/MD/SIPRISummary_260929.md'), PosixPath('/content/drive/MyDrive/Praxis/Course Geopolitics/Centaur Project/WorldState/MD/ShortTermEnergyOutlo_260929.md')]


In [ ]:
for f in md_files:
    print(f.name)

1790418308FinalSnaps_260929.md
IranData_260929.md
MissilesofIranMissil_260929.md
OilMarketReportSepte_260929.md
PressReleasePagePres_260929.md
SIPRISummary_260929.md
ShortTermEnergyOutlo_260929.md


In [ ]:
mos1.ShowMD(md_files[5])

# SOURCE: SIPRI_Summary

    **Source PDF:** `SIPRI_Summary.pdf`

    ---

    Date of publication: 2026

Armaments, Disarmament and International Security  
SIPRI YEARBOOK 2026 — Summary

© SIPRI 2026  
www.sipriyearbook.org

Stockholm International Peace Research Institute (SIPRI) is an independent international institute dedicated to research into conflict, armaments, arms control and disarmament. Established in 1966, SIPRI provides data, analysis and recommendations, based on open sources, to policymakers, researchers, media and the interested public.

THE SIPRI YEARBOOK  
SIPRI Yearbook 2026 presents a combination of original data in areas such as world military expenditure, international arms transfers, arms production, nuclear forces, armed conflicts and multilateral peace operations with state-of-the-art analysis of important aspects of arms control, peace and international security.

This booklet summarizes the contents of SIPRI Yearbook 2026 and provides samples of the data and analysis that it contains.

Contents
- Part I. International security and armed conflict, 2025
  1. Reflections on the challenges of systemic disruption and the future of peace research
  2. Global trends in armed conflict
  3. Regional security and conflict dynamics
  4. Peace processes and multilateral peace operations
- Part II. Military spending and armaments, 2025
  5. Military expenditure
  6. Arms production and military services
  7. International arms transfers
  8. World nuclear forces
  9. Proliferation and use of missiles and armed uncrewed aerial vehicles
- Part III. Non-proliferation, arms control and disarmament, 2025
  10. Nuclear disarmament, arms control, non-proliferation and security
  11. Chemical weapons and security threats
  12. Biological weapons and health security threats
  13. Conventional arms control and the regulation of inhumane weapons
  14. Artificial intelligence and international peace and security
  15. Cyber and digital technologies: Trends, incidents and governance
  16. Space security governance
  17. Dual-use and arms trade controls
- Annexes

Part I. International security and armed conflict, 2025

1. REFLECTIONS ON THE CHALLENGES OF SYSTEMIC DISRUPTION AND THE FUTURE OF PEACE RESEARCH
karim haggag, sipri director

This edition of the SIPRI Yearbook coincides with the 60th anniversary of the Institute’s founding. Having experienced and responded to six decades of tumultuous change, a new era of systemic disruption in global security brings new research challenges. This effort must now address a set of highly complex questions that include understanding the sources of the current disruption; mitigating the mounting geopolitical and human insecurities that flow from this condition; and formulating alternatives for constructing new frameworks for peace, stability and security.

The new strategic landscape
The most recent decade has fundamentally altered the strategic environment. The distinguishing feature of this current phase of great power competition relates to two overarching drivers: the resurgence of large-scale interstate war between technologically advanced states and the erosion of the United States’ alliance frameworks. Russia’s annexation of Crimea and invasion of Ukraine marked the return of an East–West divide in Europe and the collapse of hopes for integrating Russia into a post–cold war European security order. At the same time, China’s rise and its intensifying rivalry with the USA have transformed the global balance of power. In addition, the three foundations of global nuclear order have eroded: the strategic nuclear arms control framework has collapsed; the global non-proliferation regime is under stress; and US extended deterrence commitments are diminishing in credibility. The erosion of these foundations is being exacerbated by emerging disruptive military technologies and coercive competition between major states.

Rising threats to human security
Over the course of the past two decades, there has been a gradual shift away from the liberal paradigm of peacemaking that previously dominated conflict resolution efforts towards a more power-based and transactional approach. This was characterized by US ‘peace diplomacy’ under the second administration of Donald J. Trump in 2025. The retreat of multilateral peace frameworks, the severe erosion of the various constraints on the use of force and the resulting increase of conflict-related threats to human security are converging to constitute a manifold interlocking crisis.

A renewed peace research agenda
A future peace research agenda could be centred on four broad thematic issues: stabilizing great power competition; developing new arms control frameworks suited to emerging technologies and geopolitical fragmentation; understanding and addressing the human insecurity drivers of conflict; and anticipating the implications of future battlefields and new military technologies.

Peace research will need to avoid being pulled in limiting directions: towards serving narrow national security agendas, towards being absorbed into defence research or towards abandoning human security in favour of ‘hard security’ alone. Peace research needs to be preserved as an independent, holistic field whose purpose is not just to study conflict, but to help stabilize adversarial relations and ultimately support more durable forms of peace.

2. GLOBAL TRENDS IN ARMED CONFLICT

In 2025 the global landscape of armed conflict retained the form it took in 2024, with large-scale and sustained violence across multiple regions. The number of locations of armed conflict fell slightly, from 50 states in 2024 to 49 in 2025. Most of the fighting in those states continued to be internal—involving forces of the government and non-state armed groups—rather than against another state. However, the number of interstate armed conflicts doubled from three in 2024 to six in 2025 and involved at least 13 states: Afghanistan–Pakistan; Cambodia–Thailand; India–Pakistan; Iran–Israel/United States; Russia/North Korea–Ukraine; and the Democratic Republic of the Congo–Rwanda. The war in Gaza fell into something of a grey zone as it can be classified as part of the broader Palestinian conflict between a state (Israel) and an actor aspiring to statehood.

The overall estimated number of conflict-related fatalities reduced slightly (by 4.2 per cent) from 249 000 in 2024 to 238 000 in 2025 but this was still higher than in any other year in the period 2018–25 (the period for which consistent data is available). There were five major armed conflicts in 2025 (i.e. involving over 10 000 conflict-related fatalities), one more than in 2024. The number of high-intensity conflicts (i.e. involving 1000–9999 conflict-related fatalities) reduced from 23 in 2024 to 20 in 2025.

Precision-strike weapons, artificial intelligence-assisted targeting, autonomous weapon systems, swarms of uncrewed aerial vehicles (UAVs, or drones) and cyber operations are now being routinely used in armed conflicts, raising many ethical and legal questions, especially about human oversight. Violations of international humanitarian law are increasing, with reports of child soldier recruitment, sexual violence, starvation tactics and attacks on health services showing a worrying rise in prohibited methods of warfare. Forced displacement also remains a large-scale humanitarian challenge. By mid 2025, around 117.3 million people were displaced, reflecting the human cost of the conflicts.

The global trends in armed conflict visible in 2025 suggest that the norm against territorial conquest is under severe strain, with annexations and occupations in Ukraine, the Middle East, the South China Sea and elsewhere testing the post-World War II principle that territory cannot be acquired by force. If left unchecked, the erosion of the prohibition on the use of force will return geopolitics to a raw contest of military power.

Major armed conflicts and estimated conflict-related fatalities, 2025

| Location         | Estimated conflict-related fatalities, 2025 | Change (%), 2024–25 |
|------------------|--------------------------------------------:|---------------------:|
| Ukraine          | 78 350 a                                   | 6                    |
| Sudan            | 18 447                                     | 11                   |
| Israel–Palestine | 16 933                                     | –44                 |
| Myanmar          | 15 468                                     | –23                 |
| Nigeria          | 12 096                                     | 22                   |

Note: Based on data from the Armed Conflict Location and Event Data (ACLED) project. Other public sources suggest higher fatality rates in some cases. Major armed conflicts involve over 10 000 conflict-related fatalities.  
a Estimated fatalities in Ukraine include fatalities of Russians and North Koreans fighting there. According to ACLED data, there were a further 1856 conflict-related fatalities in Russia in 2025.

3. REGIONAL SECURITY AND CONFLICT DYNAMICS

Regional security and conflict dynamics are defined by a complex mix of threats and challenges to state and human security. Across all major world regions in 2025 there was a pattern of escalating armed conflict, great power competition, fragmenting world order and climate change-driven insecurity.

The Americas
Key human and state security trends in the Americas visible in 2025 included the attempted reassertion of regional hegemony by the United States amid great power competition with China in the region. This involved increasing economic and political pressure across the Americas and intervening militarily in Venezuela. Another notable trend was that criminal groups continued to challenge the state’s monopoly on violence and control of territory in several countries in the region.

Asia and Oceania
In 2025 the diverse security landscape in Asia and Oceania continued to be marked by intensifying strategic competition between China and the USA. Other trends in the region included the ongoing build-up of arms in many states and widespread domestic instability often resulting in democratic erosion or political volatility. Several critical regional flashpoints remained at risk of escalating into major interstate armed conflicts; for example, an unusually severe military crisis erupted between India and Pakistan in May 2025.

Europe
The security picture in Europe in 2025 was shaped primarily by the continuing Russia–Ukraine war, which meant that it was the region with the highest number of conflict-related fatalities. In many European countries, the war has led to a refocusing of state priorities on hard power, leading to sharp increases in military expenditure, and has amplified political fragmentation and great power competition.

The Middle East and North Africa
The Middle East and North Africa is a region characterized by deep-seated instability. In 2025 it hosted roughly one fifth of forcibly displaced people worldwide and faced acute food and water insecurity. While a formal ceasefire that took effect in October halted large-scale military offensives in the war in Gaza, subsequent breaches were frequent and severe. The war has resulted in grave violations of international law and a humanitarian catastrophe, with 90 per cent of Gaza’s population displaced and reported Palestinian deaths totalling over 71 000 by the end of 2025. The Iran–Israel and Saudi Arabia–United Arab Emirates rivalries also fuelled regional instability and armed conflict during the year.

Sub-Saharan Africa
Sub-Saharan Africa continues to face a complex web of interconnected human and state security challenges. These include extreme poverty, armed conflict, terrorist threats, climate change vulnerability, weak governance and intense geopolitical competition as states outside the region scramble for influence and resources. Sub-Saharan Africa hosted more than 45 per cent of the global total of internally displaced people in 2025 and an estimated 167 million Africans faced acute food insecurity—a record high.

4. PEACE PROCESSES AND MULTILATERAL PEACE OPERATIONS

Peace processes became more fragmented, transactional and coercive in 2025. There was less emphasis on comprehensive, inclusive settlements and more on rapid deals, ceasefires and leader-driven bargaining. This approach achieved some outline agreements, but these were often fragile, disputed or incomplete, and prioritized economic and strategic interests over durable reconciliation.

Increasing geopolitical rivalries have had a huge impact on multilateral peace operations in recent years. The United States is no longer willing to sustain the multilateral system and in 2025 took significant action to withdraw from, defund and challenge the efficacy of various United Nations bodies. China and the European Union are unwilling or unable to replace the USA in this role. In this context of multilateral inaction, middle powers—including Brazil, Qatar, Saudi Arabia, Türkiye and the United Arab Emirates—have gained increased manoeuvring space to substitute for, or complement, the multilateral security frameworks while actively advancing their own national interests.

Peace processes in 2025
There were partial breakthroughs and agreements in some major or long-running conflicts during 2025, including Armenia–Azerbaijan, the Democratic Republic of the Congo–Rwanda, Kyrgyzstan–Tajikistan and the war in Gaza. However, implementation gaps and renewed violence remain major risks.

In 2025 the UN Security Council endorsed the establishment of US President Donald J. Trump’s proposed Board of Peace as a new international transitional body to supervise governance in Gaza. However, the board’s founding Charter contains far more ambitious aims that are potentially in competition with those of the UN.

Multilateral peace operations
Multilateral peace operations faced political and financial strain in 2025, as well as declining consensus and mission withdrawal, signalling a challenging outlook for traditional peacebuilding. There were 58 active peace operations in 2025, which was 3 fewer than in the previous year, meaning that the number fell below 60 for the first time since 2016. Two started during 2025: the African Union (AU) Support and Stabilization Mission in Somalia (AUSSOM), which succeeded the AU Transition Mission in Somalia (ATMIS); and the Gang Suppression Force (GSF) in Haiti. Four had closed by the end of the year: the Multinational Security Support Mission in Haiti (MSS); the Personal Representative of the Organization for Security and Co-operation in Europe (OSCE) Chairperson-in-Office on the Conflict Dealt with by the OSCE Minsk Conference (PRCIO); the Southern African Development Community (SADC) Mission in the Democratic Republic of the Congo (SAMIDRC); and the UN Assistance Mission in Iraq (UNAMI).

The number of personnel deployed to multilateral peace operations globally has fallen by nearly 49 per cent during the past decade, dropping from 152 803 in December 2016 to 78 633 in December 2025. The overall year-on-year decrease in personnel numbers of 15 818 in 2025 can primarily be attributed to reductions in sub-Saharan Africa, where most peace operation personnel were deployed.

In the past decade, the UN’s peacekeeping budget has declined (in nominal terms) from about US$7.9 billion to just under $5.4 billion, but it continued to be the main organization deploying multilateral peace operations in 2025. It accounted for 18 operations and 67 per cent of all personnel deployed as of December 2025. However, most peace operations were deployed by regional organizations or alliances, which led 34 multilateral peace operations in 2025 (3 fewer than in 2024). Ad hoc coalitions of states conducted 6 multilateral peace operations in 2025, which was the same number as in 2024.

Number of international personnel in multilateral peace operations, by year (2016–2025) — totals (as extracted)

| Year | Total international personnel |
|------:|------------------------------:|
| 2016 | 110 181 |
| 2017 | 107 189 |
| 2018 | 103 528 |
| 2019 | 97 519 |
| 2020 | 94 201 |
| 2021 | 88 823 |
| 2022 | 92 594 |
| 2023 | 76 372 |
| 2024 | 69 913 |
| 2025 | 55 229 |

Region breakdown (numbers as extracted; regional order below)
Regions (order used in the extracted data): Sub-Saharan Africa; Middle East and North Africa; Europe; Americas; Asia and Oceania

Data rows (year followed by numbers as extracted):
- 2016: 14 351; 8 832; 5 464; 13 975
- 2017: 14 949; 8 649; 1 606; 15 467
- 2018: 14 408; 8 126; 1 433; 17 296
- 2019: 15 082; 7 819; 275; 17 086
- 2020: 14 615; 8 063; 304; 9 941
- 2021: 14 289; 8 108; 301; 337
- 2022: 14 206; 7 567; 306; 311
- 2023: 14 283; 9 215; 377; 321
- 2024: 14 498; 8 898; 828; 314
- 2025: 12 005; 9 755; 1 336; 308

Part II. Military spending and armaments, 2025

5. MILITARY EXPENDITURE

Estimated global military expenditure rose for the 11th consecutive year in 2025, to reach US$2.9 trillion—or 2.5 per cent of the world’s gross domestic product (GDP)—pushing world spending to the highest level recorded by SIPRI. The slower pace of increase in 2025 (+2.9 per cent in real terms, compared with +9.7 per cent in 2024) largely resulted from a policy shift by the United States to scale back its spending on military aid. Most of the rest of the world kept on increasing military expenditure at a fast pace.

In 2025 total regional military expenditure rose in Africa (+8.5 per cent to $58.2 billion), Asia and Oceania (+8.1 per cent to $681 billion) and Europe (+14 per cent to $864 billion), but fell in the Americas (–6.6 per cent to $1065 billion) and was largely unchanged in the Middle East ($218 billion). Spending increased in all regions over the decade 2016–25. The average military burden in Europe continued to rise, from 2.8 per cent in 2024 to 3.2 per cent in 2025, more than double the level in 2016.

The USA remained by far the largest military spender in the world; its expenditure of $954 billion in 2025 (7.5 per cent lower than in 2024) accounted for 33 per cent of total global spending. China, the second largest military spender ($336 billion), accounted for 12 per cent of the world total, while Russia, the third largest spender ($190 billion), accounted for 6.6 per cent.

Taken together, the 15 largest military spenders accounted for 80 per cent of the world’s military expenditure in 2025 ($2304 billion). Many of the 15 largest military spenders were involved—either directly or indirectly—in the wars in Gaza and Ukraine in 2025, including Israel, Russia and Ukraine as well as the USA and several other members of the North Atlantic Treaty Organization (NATO). With military spending equivalent to 40 per cent of its GDP, Ukraine remained the country with the largest military burden in the world in 2025. Russia’s military burden was 7.5 per cent in 2025.

In June 2025 the NATO members agreed to a new military spending target of 5.0 per cent of GDP to be reached by 2035, significantly elevating the previous benchmark of 2.0 per cent agreed in 2014. In September 2025 the UN secretary-general, António Guterres, released The Security We Need, a landmark report calling for a rebalancing of military spending in the interest of sustainable development.

Top 15 military spenders, 2025

| Rank | Country | Spending (US$ b.) | Change (%), 2024–25 |
|-----:|---------|-------------------:|--------------------:|
| 1 | United States | 954 | –7.5 |
| 2 | China | [336] | 7.4 |
| 3 | Russia | [190] | 5.9 |
| 4 | Germany | 114 | 24 |
| 5 | India | 92.1 | 8.9 |
| 6 | United Kingdom | 89.0 | –2.0 |
| 7 | Ukraine | [84.1] | 20 |
| 8 | Saudi Arabia | [83.2] | 1.4 |
| 9 | France | 68.0 | 1.5 |
| 10 | Japan | 62.2 | 9.7 |
| 11 | Israel | 48.3 | –4.9 |
| 12 | Italy | 48.1 | 20 |
| 13 | South Korea | 47.8 | 2.6 |
| 14 | Poland | 46.8 | 23 |
| 15 | Spain | 40.2 | 50 |
| **Subtotal top 15** |  | **2 304** |  |
| **World** |  | **2 887** | **2.9** |

. . = data not available or not applicable; [ ] = estimated figure.  
Note: Spending figures are in billions of US dollars, at current (2025) prices and exchange rates. Changes are in real terms, based on constant (2024) US dollars.

6. ARMS PRODUCTION AND MILITARY SERVICES

The SIPRI Top 100
The combined arms revenues of the 100 largest arms-producing and military services companies (the SIPRI Top 100) totalled US$679 billion in 2024 (the most recent year for which data is available). This was 5.9 per cent higher than in 2023. The increase reflects the demand for military equipment and services driven by heightened global security concerns, ongoing conflicts and rising military budgets. Seventy-seven of the companies in the Top 100 increased their arms revenues in 2024, including 42 that reported at least double-digit percentage growth. Given the persistence of demand and the size of companies’ backlogs, there are likely to be further increases in global arms revenues in the coming years.

The United States continued to dominate the ranking with 39 companies in the Top 100 for 2024, with total arms revenues of $334 billion (3.8 per cent more than in 2023). China was second with 8 companies in the ranking, with combined arms revenues of $88.3 billion (10 per cent less than in 2023). Due to a lack of available data, only 2 Russian companies were included in the Top 100 for 2024. Their combined arms revenues of $31.2 billion were 23 per cent higher than in 2023.

Supply chain weaponization
The industrial capacity to meet the increased demand for advanced weapon systems is being hampered by the same geopolitical tensions driving the demand. Globalization brought with it complex, international supply chains and cheaper critical materials and components. While concentrated global supply chains can maximize cost-effectiveness, they can also create chokepoint vulnerabilities. Since 2020, for example, China has steadily tightened exports of critical materials and several of the world’s largest arms companies have cited such restrictions as a major risk to their operations.

To minimize their reliance on current sources, states and companies have opted to diversify their suppliers, focusing on strategies to reconfigure global supply chains to increase resilience, reduce risks and align with political or economic allies. But shifting towards greater autarky is a long-term endeavour and not being able to leverage economies of scale means more expensive weapons and reduced purchasing power for military procurement budgets.

Top 10 arms-producing and military services companies in the world, by arms revenues, 2024

| Rank | Company | Country | Arms revenues (US$ m.) |
|-----:|---------|---------|------------------------:|
| 1 | Lockheed Martin | USA | 64 650 |
| 2 | RTX | USA | 43 600 |
| 3 | Northrop Grumman | USA | 37 850 |
| 4 | BAE Systems | UK | 33 790 |
| 5 | General Dynamics | USA | 33 630 |
| 6 | Boeing | USA | 30 550 |
| 7 | Rostec | Russia | 27 120 |
| 8 | AVIC | China | 20 320 |
| 9 | CETC | China | 18 920 |
| 10 | L3Harris Technologies | USA | 16 210 |

Note: Arms revenues are in millions of US dollars, at constant (2024) prices and exchange rates. ‘Country’ refers to the country in which the ownership and control structures of the company are located, i.e. the location of a company’s headquarters.

7. INTERNATIONAL ARMS TRANSFERS

The volume of international transfers of major arms in the five-year period 2021–25 was the highest since the end of the cold war, having risen by 9.2 per cent compared with 2016–20. There are strong indications that demand for major arms is growing, which will probably lead to further increases in global arms flows in the coming years.

Conflicts, tensions and arms transfers
Armed conflicts and geopolitical tensions are the main drivers of arms procurement for many states. Most of the world’s largest recipients of major arms in 2021–25 used imported arms in military combat operations in that period. Arms-exporting states are often directly or indirectly affected by the conflicts or tensions driving demand in importing states, which partly explains why they are willing to supply arms, even when this seems to contradict their stated arms export policies.

During 2021–25 Israel was involved in a multifront war stemming from its large-scale military offensive in Gaza that began in October 2023. Israel continued to receive major arms from supplier states throughout the period despite growing concerns in some states about the high civilian death toll in Gaza and Israeli violations of international law. The United States was the largest supplier to Israel in 2021–25 (accounting for 68 per cent of Israeli arms imports), followed by Germany (31 per cent).

Suppliers of major arms
SIPRI has identified 66 states as suppliers of major arms in 2021–25, but most were minor exporters. The 25 largest suppliers accounted for 98 per cent of the total volume of exports, and the top 5—the USA, France, Russia, Germany and China—accounted for 70 per cent.

The USA’s arms exports grew by 27 per cent between 2016–20 and 2021–25, increasing its share of global arms exports from 36 per cent to 42 per cent. Known plans for future deliveries strongly indicate that the USA will remain by far the world’s largest arms exporter—a position leading to anxieties of dependence for some of its main clients and allies. In contrast, Russia’s arms exports more than halved between 2016–20 and 2021–25 (–64 per cent), giving it a global share of 6.8 per cent—a level far below any previous five-year period in its history or in any previous five-year period since 1950 for its predecessor, the Soviet Union.

Taken together, the arms exports of the 27 current European Union member states went up by 36 per cent between 2016–20 and 2021–25. They accounted for 28 per cent of total global arms exports in 2021–25, which was two thirds the volume of US arms exports in the same period, but four times higher than Russia’s export volume and five times higher than China’s. Arms exports by France rose by 21 per cent between 2016–20 and 2021–25 and accounted for 9.8 per cent of the global total, while arms exports by Germany increased by 15 per cent, giving it a 5.7 per cent share.

Recipients of major arms
SIPRI has identified 162 states as recipients of major arms in 2021–25. The five largest recipients were Ukraine, India, Saudi Arabia, Qatar and Pakistan, which together accounted for 35 per cent of total arms imports in the period. Ukraine’s arms imports were around 100 times higher in 2021–25 than in 2016–20 and it became the world’s largest arms recipient with at least 36 states delivering major arms, mostly as aid. Arms imports by China fell by 72 per cent between 2016–20 and 2021–25 as it continued to expand its domestic design and production capabilities for major arms. Arms imports by states in Europe more than trebled between 2016–20 and 2021–25 (+210 per cent), reaching a level far higher than at any time since the end of the cold war and making Europe the largest importing region for the first time since the 1960s. Arms imports by states in the Americas also increased (+12 per cent), while imports by states in Africa (–41 per cent), Asia and Oceania (–20 per cent) and the Middle East (–13 per cent) decreased.

The trend in transfers of major arms, 1950–2025
Note: The bar graph shows the average annual volume of arms transfers for 5-year periods and the line graph shows the annual totals since 1950.

The main suppliers and recipients of major arms, 2021–25

Suppliers (global share of exports, %)

| Rank | Supplier | Exports (%) |
|-----:|----------|------------:|
| 1 | USA | 42 |
| 2 | France | 9.8 |
| 3 | Russia | 6.8 |
| 4 | Germany | 5.7 |
| 5 | China | 5.6 |
| 6 | Italy | 5.1 |
| 7 | Israel | 4.4 |
| 8 | UK | 3.4 |
| 9 | South Korea | 3.0 |
| 10 | Spain | 2.3 |

Recipients (global share of imports, %)

| Rank | Recipient | Imports (%) |
|-----:|-----------|------------:|
| 1 | Ukraine | 9.7 |
| 2 | India | 8.2 |
| 3 | Saudi Arabia | 6.8 |
| 4 | Qatar | 6.4 |
| 5 | Pakistan | 4.2 |
| 6 | Japan | 3.9 |
| 7 | Poland | 3.6 |
| 8 | USA | 2.9 |
| 9 | Kuwait | 2.8 |
| 10 | Australia | 2.8 |

Imports of major arms, by region (2021–25)

| Recipient region | Global share (%), 2021–25 | Change (%) in volume of imports from 2016–20 to 2021–25 |
|------------------|--------------------------:|------------------------------------------------------:|
| Africa | 4.3 | –41 |
| Americas | 5.6 | 12 |
| Asia and Oceania | 31 | –20 |
| Europe | 33 | 210 |
| Middle East | 26 | –13 |

Part II continued: 8. WORLD NUCLEAR FORCES

At the start of 2026, nine states—the United States, Russia, the United Kingdom, France, China, India, Pakistan, North Korea and Israel—together possessed approximately 12 187 nuclear weapons, of which 9 745 were in military stockpiles and considered to be potentially operationally available. An estimated 4 012 of these stockpiled warheads were deployed with operational forces, with just over half thought to be kept in a state of high operational alert on ballistic missiles (2100–2200 warheads).

Overall, the number of nuclear warheads in the world continues to decline, but this is only due to the USA and Russia dismantling retired warheads. Notably, the number of warheads being dismantled annually appears to be decreasing and it seems likely that the rate at which retired warheads are dismantled will soon be outpaced by the rate at which new warheads enter global stockpiles. Low levels of transparency hinder assessments of the status of nuclear arsenals and states have become even more secretive about their nuclear weapons over the past few years. This is partly due to the degradation of arms control agreements that included transparency measures, such as the 2010 Treaty on Measures for the Further Reduction and Limitation of Strategic Offensive Arms (New START).

Nuclear weapon modernization trends
All the nine nuclear-armed states continued to strengthen their nuclear arsenals in 2025 and some deployed new nuclear-armed or nuclear-capable weapon systems during the year. The USA and Russia together possess almost 86 per cent of all nuclear warheads and both have extensive nuclear modernization programmes under way. China is also in the middle of a significant modernization and expansion of its nuclear arsenal, which is estimated to have increased from 600 to up to 620 warheads during the year.

The modernization and further deployment of nuclear weapons raise potential risks related to nuclear escalation. Two of the most destabilizing developments are nuclear–conventional entanglement—the increasing overlap between dual-capable nuclear and non-nuclear (conventional) military capabilities, especially missiles—and the proliferation of multiple independently targetable re-entry vehicles. In recent years, submarine-based nuclear weapon delivery systems have also been proliferating, especially in the four nuclear-armed states in the Indo-Pacific.

Nuclear doctrines and nuclear sharing
States also appear to be changing to reflect this new environment, with several states expanding the role of nuclear weapons in their security strategies and lowering the threshold for potential use. There were notable developments related to nuclear sharing and coordination in 2025, including a new coordination agreement between France and the UK.

Global nuclear weapon inventories, January 2026 (map and figures summarized)

Estimated warheads by country (January 2026)

| Country | Military stockpile a | Deployed b | Total inventory c |
|---------|---------------------:|-----------:|------------------:|
| USA | 1 770 | 3 700 | 5 042 |
| Russia | 1 796 | 4 400 | 5 420 |
| UK | 120 | 225 | 225 |
| France | 280 | 290 | 370 |
| China | 34 | 620 | 620 |
| India | 12 | 190 | 190 |
| Pakistan | – | 170 | 170 |
| North Korea | – | 60 | 60 |
| Israel | – | 90 | 90 |
| **Total** | **4 012** | **9 745** | **12 187** |

– = nil or a negligible value.  
Note: All estimates are approximate. SIPRI revises its world nuclear forces data each year based on new information and updates to earlier assessments. Countries are ordered by date of first known nuclear test; however, there is no conclusive open-source evidence that Israel has tested its nuclear weapons.

Definitions:
- a ‘Military stockpile’ refers to all deployed warheads as well as warheads in central storage (but not retired warheads) that could potentially be deployed after some preparation.
- b ‘Deployed warheads’ are those placed on missiles or located on bases with operational forces.
- c ‘Total inventory’ includes stockpiled warheads as well as retired warheads awaiting dismantlement.

Fissile material notes
The explosive material utilized in nuclear weapons is fissile material, either highly enriched uranium (HEU) or separated plutonium. China, France, Pakistan, Russia, the United Kingdom and the United States have produced both HEU and plutonium for use in their nuclear weapons; and India and Israel have produced mainly plutonium. North Korea has produced plutonium for use in nuclear weapons but is believed to be producing HEU for nuclear weapons as well. All states with a civilian nuclear industry are capable of producing fissile materials.

The International Panel on Fissile Materials compiles information on global stocks of fissile materials.

Global stocks of fissile materials, 2025

| Material | Global stocks (tonnes), 2025 | In or available for weapons | Not directly available for weapons |
|---------:|----------------------------:|---------------------------:|----------------------------------:|
| Highly enriched uranium (HEU) | 1 240 | 1 100 | Unsafeguarded: 130; Safeguarded/monitored: 10 |
| Separated plutonium | 565 | 140 | Unsafeguarded: 265; Safeguarded/monitored: 160 |

Part II continued: 9. PROLIFERATION AND USE OF MISSILES AND ARMED UNCREWED AERIAL VEHICLES

The demand for and proliferation of both conventional and dual-capable missile systems, as well as a wide variety of armed uncrewed aerial vehicles (UAVs, or drones), are increasing. The systems are being produced and exported at much higher rates than previously, driven by ongoing conflicts and by states seeking to replenish or expand their arsenals. Advances in guidance, propulsion and payload integration are progressively making one-way attack UAVs more like short-range missiles, adding another layer of complication to efforts to regulate such systems.

Use in armed conflicts
Missiles and armed UAVs were used in armed conflicts during 2025 in a wide variety of strike roles. These included long-range precision-strike missions against adversaries’ infrastructure and strategic targets deep inside their territory; saturation salvo attacks seeking to overwhelm air and missile defences; UAV swarm and first-person-view attacks as part of battlefield operations; and intricately planned operations using attack UAVs deployed from within an adversary’s territory to strike strategic targets or assassinate high-ranking military personnel.

The fast pace of development of armed UAVs across the entire spectrum of such systems as well as their affordability and versatility represent a game-changing trend that is altering the nature of warfare, including intelligence, surveillance and reconnaissance (ISR) operations. The rapid proliferation of UAVs and the surge in their use in armed conflict have spurred a parallel arms race in countermeasures.

Regulatory mechanisms
Arms and export control frameworks in this area are already lagging behind, with existing instruments such as the Hague Code of Conduct against Ballistic Missile Proliferation, the Missile Technology Control Regime and the Wassenaar Arrangement on Export Controls for Conventional Arms and Dual-use Goods and Technologies covering only parts of the emerging threat spectrum. There is no dedicated, binding instrument governing armed UAVs, and recent multilateral forums have struggled to achieve consensus, especially in the context of the Russia–Ukraine war.

States are shifting towards more versatile, autonomous, AI-integrated platforms that can perform multiple roles (e.g. strike, ISR or swarm operations), while also investing in layered ‘drone wall’ concepts and homeland missile-defence projects (e.g. the US Golden Dome) to address evolving threats. Many types of missile and armed UAV operations taking place in conflict zones violate principles of distinction and proportionality under existing laws of armed conflict, and the push towards fully autonomous lethal UAV systems raises questions about accountability and compliance with international humanitarian law.

Operation Spider’s Web
Utilizing over 100 attack drones smuggled into Russia over several months, Operation Spider’s Web was a coordinated assault by Ukraine on multiple airbases deep inside Russia on 1 June 2025. A total of 41 Russian aircraft reportedly suffered damage, including around a third of Russia’s fleet of nuclear-capable bombers. The attack forced Russia to disperse its strategic bombers and make major changes to its military logistics.

Part III. Non-proliferation, arms control and disarmament, 2025

10. NUCLEAR DISARMAMENT, ARMS CONTROL, NON-PROLIFERATION AND SECURITY

The deterioration in the global nuclear order continued in 2025. Poor relations and lack of strategic dialogue among major nuclear-armed states continued to impede progress in disarmament and arms control. At the end of the year there was little indication that negotiations for any arms control framework involving China, Russia and the United States were imminent, in any format. This was despite a partial thaw in the relations between the two states with the biggest nuclear arsenals—Russia and the USA—and the approaching expiration of the bilateral 2010 Treaty on Measures for the Further Reduction and Limitation of Strategic Offensive Arms (New START).

Geopolitical tensions and regional conflicts highlighted the risk of nuclear weapon use as well as the possibility of uncontrolled release of radioactive materials resulting from attacks on civilian nuclear infrastructure. Although the seriousness of some of these risks and the importance of nuclear risk reduction were broadly acknowledged in multilateral forums, this did not translate into concrete action.

Attacks on nuclear installations in Iran
The global non-proliferation regime suffered a severe blow in June 2025 when Israel attacked Iran, with the USA later joining Israel’s attacks against Iranian nuclear facilities in a 12-day war. Although framed in terms of achieving the goal of non-proliferation, the attacks—condemned by several legal experts and states as a clear transgression of international law—undermined that goal by precipitating an end to continuous monitoring of Iranian nuclear facilities by the International Atomic Energy Agency. The war demonstrated the vulnerability of non-nuclear weapon states to aggression by nuclear-armed states and served to normalize the ‘preventive’ use of force.

Multilateral processes
The 12-day war and the repeated attacks on Ukrainian nuclear power plants are likely to have an impact on the 2026 Review Conference of the 1968 Treaty on the Non-Proliferation of Nuclear Weapons (NPT). These negative trends come on top of concerns over the possible resumption of nuclear testing, the expiry of New START in February 2026 and other disagreements among NPT states parties—mostly linked to the lack of progress on nuclear disarmament—that are likely to place further strain on the discussions at the conference.

States parties of the 2017 Treaty on the Prohibition of Nuclear Weapons (TPNW) were unified in taking a strong position against nuclear deterrence and their continuing commitment to the previously agreed Vienna Action Plan. However, by the end of 2025, the prospects for building meaningful political momentum with nuclear-armed states for any new risk-reduction measures, let alone steps towards disarmament, remained highly unlikely.

The continuing erosion of international norms on nuclear disarmament, non-proliferation and arms control cast a shadow over the 80th anniversary of the 1945 bombings of Hiroshima and Nagasaki. According to the mayor of Hiroshima, Kazumi Matsui, these and other concerning trends “flagrantly disregard the lessons the international community should have learned from the tragedies of history”.

11. CHEMICAL WEAPONS AND SECURITY THREATS

Chemical weapons are prohibited by the 1993 Chemical Weapons Convention (CWC), which entered into force in 1997 and had 193 states parties as of December 2025. The Organisation for the Prohibition of Chemical Weapons (OPCW) oversees the implementation of the CWC, verifies chemical weapon disarmament and helps prevent the re-emergence of chemical weapons. The OPCW successfully completed the disarmament of all declared chemical weapon stockpiles worldwide in 2023 and its focus has shifted to the prevention of the re-emergence of chemical weapons.

Allegations of and reactions to chemical weapon use
While the CWC is one of the most successful disarmament treaties, it has faced significant challenges and compliance concerns, mainly concerning Syria and Russia. Syria acceded to the CWC in 2013 and subsequently declared a chemical weapon programme that was thought to have been eliminated under OPCW verification by 2016. However, OPCW investigations later proved that Syria had been in non-compliance and had used chemical weapons on several occasions since 2014. Over the past decade, OPCW member states have sought to bring Syria into compliance without success. However, the situation in Syria shifted significantly in 2025 after the new government pledged to cooperate fully with the OPCW to dismantle the remainder of the Syrian chemical weapon programme. Allegations of chemical weapon use in Sudan and Ukraine continue to give rise to concerns.

Chemical weapon control and disarmament
The OPCW has been affected by the conflict in Ukraine in several ways, including persistent polarization in the policymaking organs and being subject to continuing disinformation campaigns. However, the improving situation in Syria contributed to less confrontational exchanges in 2025 and consensus decisions on previously more contested issues, such as the OPCW’s budget. The OPCW also continued its routine activities, including in the areas of verification, international cooperation and assistance, and monitoring technological developments relevant to the CWC, notably in the field of artificial intelligence. The OPCW Centre for Chemistry and Technology, which opened in 2023, has played an important role in supporting the OPCW’s work with its technical capabilities and laboratory capacity.

In November 2025 the OPCW member states appointed Ambassador Sabrina Dallafior of Switzerland as the next director-general of the OPCW. She will take office in July 2026.

Alleged chemical weapon use in Sudan and Ukraine
Technical assistance visits by the Organisation for the Prohibition of Chemical Weapons in 2025 confirmed the presence of a riot control agent on battlefield sites in Ukraine, and several states requested clarification from Russia regarding these findings. The clarification procedure was also invoked in 2025 to address the alleged use of chemical weapons in Sudan.

12. BIOLOGICAL WEAPONS AND HEALTH SECURITY THREATS

Biological weapons comprise bacteria, viruses or toxins disseminated through a delivery mechanism to inflict harm and are prohibited under international law. The principal legal instrument banning biological warfare is the 1972 Biological and Toxin Weapons Convention (BWC). The BWC is moving towards universality, with Comoros and Kiribati acceding to the convention in 2025, taking the number of states parties to 190.

The wider biological warfare regime includes the 1925 Geneva Protocol, which prohibits the use of chemical and biological weapons in war, and a growing array of other measures, including export control regimes like the Australia Group, the United Nations secretary-general’s mechanism for investigation of alleged use of chemical and biological weapons, and UN Security Council Resolution 1540. These broader measures all serve to bolster aspects of the prohibition and prevention of biological weapons.

Biological weapon disarmament and non-proliferation
There were two significant anniversaries in the history of biological weapon prohibition in 2025: the centenary of the 1925 Geneva Protocol and the 50th anniversary of the entry into force of the BWC. The anniversaries reaffirmed the strength of the norm against biological weapons while underscoring the difficulty of translating that norm into effective institutional capacity.

The 2025 sessions of the working group on strengthening the BWC were defined by a shift from conceptual debate to drafting concrete text, although procedural deadlocks at the end of the year prevented formal adoption of a final substantive report. Similarly, the 2025 Meeting of States Parties of the BWC was largely defined by procedural deadlock and the inability to reach consensus on substantive outcomes. At the First Committee of the UN General Assembly, the United States encouraged countries to join a USA-led artificial intelligence (AI) verification initiative to enhance the enforcement and monitoring of the BWC.

Technological advancements and risks
Advances in AI, synthetic biology and laboratory automation are expanding the scale, speed and accessibility of the life sciences, intensifying both their beneficial and destabilizing potentials. These developments place new demands on governance systems that were largely designed for a different technological era. The challenge for arms control and biosecurity frameworks revolves around institutional adaptation, implementation and trust-building across increasingly complex scientific ecosystems.

Global health security dynamics
Infectious disease threats, pandemic preparedness and international health governance intersected with issues of transparency, trust and geopolitical competition in 2025. The United States withdrew from the World Health Organization (WHO); a new Pandemic Agreement was adopted by WHO member states; and a WHO advisory group concluded that the investigations into the origins of Covid-19 remain inconclusive.

13. CONVENTIONAL ARMS CONTROL AND THE REGULATION OF INHUMANE WEAPONS

The main multilateral treaty for regulating inhumane weapons is the 1981 Convention on Certain Conventional Weapons (CCW Convention). There are also separate conventions on anti-personnel mines and cluster munitions. Other categories of conventional weapon that raise humanitarian concerns are dealt with by other legal and political processes.

Withdrawals from cluster munitions and anti-personnel mines conventions
A key development in 2025 was the unprecedented withdrawals from the 2008 Convention on Cluster Munitions (CCM) and the 1997 Anti-Personnel Mine (APM) Convention. Lithuania’s withdrawal from the CCM, which went into effect on 6 March 2025, marked the first time that a state had withdrawn from a humanitarian arms control treaty. Shortly after, five European states—Estonia, Finland, Latvia, Lithuania and Poland—announced their intent to withdraw from the APM Convention, citing security concerns related to the Russia–Ukraine war. The withdrawal notices for Estonia, Latvia and Lithuania took effect on 27 December 2025, while Poland’s took effect on 10 January 2026 and Finland’s on 20 February 2026.

Explosive weapons in populated areas
The use of explosive weapons in populated areas (EWIPA) continued to be widespread in major armed conflicts in 2025, with particularly devastating effects in Gaza, Sudan and Ukraine. A 2022 political declaration that seeks to address the humanitarian consequences of the use of EWIPA has been endorsed by 90 states. The second follow-up conference in 2025 concluded with a renewed collective commitment to translate political promises into concrete military and humanitarian action.

Tension between the humanitarian imperative and military necessity
Efforts to preserve or strengthen the achievements of multilateral conventional disarmament and arms control and to spread norms that reduce the human cost of weapons are clearly under stress. In general, many states are now valuing the pursuit of politico–military advantage over arms control. Amid growing strategic competition, these states are increasingly unwilling to agree to arms control restraints and transparency measures in the belief that these could yield advantages to adversaries. The frameworks designed to protect human life and dignity are under serious pressure at a time when civilians face heightened risks during armed conflict and at a time when international law more broadly is under threat.

Use of inhumane conventional weapons in 2025

| Weapon category | State use |
|-----------------|-----------|
| Anti-personnel mines | Cambodia a; Iran; North Korea; Myanmar; Russia; Ukraine a |
| Cluster munitions | Myanmar; Iran a; Russia; Thailand a; Ukraine |
| EWIPA (explosive weapons in populated areas) | Widespread use, including in major armed conflicts in Gaza, Sudan and Ukraine |
| Incendiary weapons | Russia; South Sudan; Ukraine |

EWIPA = explosive weapons in populated areas.  
a Alleged use.

14. ARTIFICIAL INTELLIGENCE AND INTERNATIONAL PEACE AND SECURITY

Advances in artificial intelligence (AI) are poised to bring enormous benefits, but they can also create or exacerbate existing threats to international peace and security. In recent years, states have increasingly acknowledged the need to manage these complex risks—stemming from both civilian and military AI—through the establishment of new forums and initiatives. Discussions and work related to these forums and initiatives continued in 2025.

Military AI
For the past decade the international policy conversation on military uses of AI has mostly focused on autonomous weapon systems (AWS), commonly characterized as weapon systems that, once activated, can select and engage targets without human intervention. Since 2023, however, the conversation has expanded to other military uses of AI such as in targeting, planning and intelligence analysis. Reported uses of AI in major armed conflicts in 2025, especially Gaza and Ukraine, led policymakers to pay particular attention to so-called AI-enabled decision support systems.

Discussion on AWS has historically primarily taken place in Geneva under the framework of the 1981 CCW Convention, but in 2025 the United Nations General Assembly started a dedicated discussion track in New York. In addition, the UN secretary-general, António Guterres, published his first-ever report on AI in the military domain, and the UN General Assembly agreed to convene informal exchanges in 2026 on the peace and security implications of military AI. The three-year mandate of the CCW Convention’s group of governmental experts concludes in 2026 and next steps regarding the regulation of AWS will need to be agreed—not least whether states can start formally negotiating the framework and content of a new governance instrument.

Civilian AI
Efforts to understand the implications of civilian AI for peace and security also continued to evolve during the year. Developments included an agreement by UN member states to establish an Independent International Scientific Panel on AI; the creation of a multistakeholder Global Dialogue on AI Governance; and the AI Action Summit in Paris. In addition, the European Union and several states leading the development of AI—including China and the United States—transitioned from foundational laws or voluntary AI principles (mostly drafted in 2023–24) to adopting or enacting formal, enforceable governance processes and new detailed compliance guidelines and directives.

Key AI governance documents adopted in 2025
- Code of Practice for General-purpose AI Models: Adopted by the European Union (EU) in July.  
- EU AI strategy documents: The AI Continent Strategy (released in April) and the Apply AI Strategy (released in October).  
- China’s Global AI Governance Action Plan: Announced at the World AI Conference in July, this 13-point roadmap promotes international cooperation on AI safety standards and technical norms.  
- The United States’ AI Action Plan: Announced in July, the plan outlines national priorities for federal AI research and sector-specific safety benchmarks for critical infrastructure and healthcare.

15. CYBER AND DIGITAL TECHNOLOGIES: TRENDS, INCIDENTS AND GOVERNANCE

The cyber domain and its impact on geopolitics is constantly evolving. The divide between the cyber activities of national governments and criminal actors continues to blur. Cyber operations are present in multiple conflicts alongside kinetic military action. Ransomware activities are affecting public and private data, critical infrastructure and politics on an unprecedented scale. AI has become the defining variable in cybersecurity, empowering adversaries to discover vulnerabilities, launch attacks and evade detection, while simultaneously providing defenders with new capacity to monitor networks, identify anomalies and respond.

Against this backdrop, multiple ongoing diplomatic processes aim to create new governance instruments and frameworks, enhance regional cooperation or address specific cyber threats.

Cyber trends
In a time of accelerating technological change and geopolitical volatility, developments in cyber capabilities in 2025 were both a catalyst for progress and a source of vulnerability. Cyber operations continued to evolve as an integral component of modern conflict over the year. Notable examples included Russia and Ukraine engaging in widespread and persistent cyber operations; India and Pakistan overtly integrating cyber operations into armed conflict for the first time when an unusually severe military crisis erupted in May 2025; and Iran and Israel employing coordinated digital retaliation during their confrontation.

Beyond active conflict zones, sophisticated espionage campaigns in 2025 demonstrated how adversaries may be pre-positioning for potential future conflicts while harvesting sensitive intelligence. Cybercrime also continued to evolve in both scale and sophistication, with ransomware and other attacks exploiting supply chains and causing growing economic damage, alongside the rapid emergence of professional scam operations.

Cyber governance
It was also a pivotal year for global cyber governance. A United Nations open-ended working group concluded its mandate and established a new permanent Global Mechanism for responsible state behaviour in cyberspace, although persistent divisions over the pertinence of international law in this field remained unresolved. There was also continued momentum in initiatives addressing cyber intrusion capabilities and ransomware. In addition, the UN Convention against Cybercrime opened for signature in October 2025 amid ongoing debates about balancing security with human rights.

Another notable trend in 2025 was that more states began making public attribution statements naming the alleged perpetrators of cyberattacks. States and regional organizations are also increasingly publishing interpretations of how international law applies to their conduct in cyberspace—to foster a more stable, secure and predictable cyber environment. Colombia, South Korea and Thailand published national positions in 2025, bringing the number to have done so to 35 states, along with the African Union and the European Union, which have published their own positions.

16. SPACE SECURITY GOVERNANCE

In the past decade, the number of states conducting space activities for economic, scientific and military purposes has grown considerably, and more states have adopted national and regional space policies. While outer space has been used for military purposes since the dawn of the space age, current geopolitical tensions and increased competition in space activities pose rising threats to space systems. The growing interest in ‘counterspace’ capabilities is particularly concerning, given the parallel growth in civilian space activities and overall societal dependence on space systems.

Space for military purposes
In 2025 several states accelerated their pursuit of new military space capabilities. This trend to pursue sovereign space capabilities dovetailed with new national and regional policies from China, the United States and within Europe that prioritize space security. US policy changes included plans to invest in space-based interceptors, policies for ‘space warfighting’ and steps to ‘ensure American space superiority’. Outer space also featured prominently in China’s white paper on arms control, disarmament and non-proliferation published in 2025.

France, Germany and the United Kingdom introduced new national strategies that included planned investment in counterspace capabilities, while the European Union proposed new legislation (a draft EU Space Act) aimed at securing independent access to space and expanding European industry. This development of independent European space capabilities is being spurred by growing uncertainty in transatlantic relations and perceived threats from Russia.

Multilateral discussions on space security
While multilateral discussions on space governance have been under way for decades, progress has been slow. In United Nations space security talks, progress stalled in 2025. In April the first session of the UN open-ended working group on prevention of an arms race in outer space (PAROS) in all its aspects grappled with procedural debates, although the second session in July moved to substantive talks. Tensions at the UN General Assembly First Committee worsened, with the USA and Israel opposing foundational UN space resolutions, particularly the annual resolution adopted on PAROS.

Nonetheless, opportunities exist to minimize risks in outer space through dialogue among several key actors, including China and European states. China signalled a willingness to engage in multilateral space security governance through its 2025 white paper, and there are several avenues for Europe, China and other states to build shared understandings of specific terminologies and concepts in space security. These could help to reduce risks of misperception, misunderstanding and miscalculation in the space domain.

The US Golden Dome
In 2025 the United States set out plans for a new multilayered missile defence system—including a layer of space-based interceptors—known as the Golden Dome. Total cost estimates released in 2025 ranged from the White House’s US$175 billion to independent projections exceeding $500 billion. The initiative will need to overcome formidable technical barriers, with concerns that it is unlikely to be a reliable defence against incoming missiles but may undermine strategic stability and trigger a new space arms race.

17. DUAL-USE AND ARMS TRADE CONTROLS

The set of global, multilateral and regional instruments that seek to establish and promote agreed standards for controls on the trade in military items and dual-use items remained under significant strain during 2025—because of geopolitical tensions, new and ongoing armed conflicts and rapid advances in key technological areas. States are increasingly acting unilaterally or operating through alternative frameworks when creating new controls on transfers of items or restricting transfers to destinations. However, there were no significant efforts to dismantle the existing instruments, indicating that many states continue to value them.

The Arms Trade Treaty
The number of states parties to the 2013 Arms Trade Treaty (ATT) continues to increase slowly but key arms suppliers and recipients have still not joined. In addition, there remain significant gaps in the number of initial reports and annual reports states are submitting. During 2025 the ATT continued to be a forum for difficult discussions about the more contentious arms exports, including arms transfers to Israel and those reaching the conflict parties in Sudan. States also explored ways to reinvigorate the ATT process and agreed on a mandate to elaborate a five-year strategy for the treaty by 2027.

Multilateral arms embargoes
During 2025 there were 14 United Nations embargoes and 22 European Union (EU) embargoes in force. No new multilateral arms embargoes were imposed, but the UN arms embargo on Iran, which had been suspended since 2023, was reimposed. The willingness of certain states to circumvent UN arms embargoes continued to undermine their effectiveness during 2025. For example, there was clear evidence of arms transfers to embargoed destinations in Libya and Yemen and of banned exports from North Korea to Russia. The EU expanded its arms embargoes on Russia and Belarus and sought to close off circumvention channels. However, efforts to align Western states’ arms embargoes on Russia became less coordinated during 2025 due to a change in US policy.

UN and EU arms embargoes in force, 2025

United Nations (14 embargoes)
- Afghanistan (NGF: Taliban)
- Central African Republic (partial; NGF)
- Democratic Republic of the Congo (NGF)
- Haiti (NGF)
- Iran
- Iraq (NGF)
- ISIL (Da’esh), al-Qaeda and associated individuals and entities (NGF)
- Korea, North
- Lebanon (NGF)
- Libya (partial; NGF)
- Somalia (NGF: al-Shabab)
- South Sudan (partial; NGF)
- Sudan (partial: Darfur)
- Yemen (NGF)

European Union (22 embargoes)
- Implementations of UN embargoes (11): Afghanistan (NGF: Taliban); Central African Republic (partial; NGF); Democratic Republic of the Congo (partial; NGF); Haiti (NGF); Iraq (NGF); ISIL (Da’esh), al-Qaeda and associated individuals and entities (NGF); Korea, North; Lebanon (NGF); Libya (partial; NGF); Somalia (NGF: al-Shabab); Yemen (NGF)
- EU arms embargoes with broader coverage than their UN counterparts (3): Iran; South Sudan; Sudan
- Embargoes with no UN counterpart (8): Belarus; China; Egypt; Myanmar; Russia; Syria; Venezuela; Zimbabwe

ISIL = Islamic State in Iraq and the Levant; NGF = non-governmental forces; partial = embargo allows transfers of arms to the government of the target state provided that certain conditions have been met.

Export control regimes
The repercussions of the Russia–Ukraine war continued to affect the work of the multilateral export control regimes—the Australia Group (on chemical and biological weapons), the Missile Technology Control Regime, the Nuclear Suppliers Group, and the Wassenaar Arrangement (WA) on Export Controls for Conventional Arms and Dual-use Goods and Technologies. Despite these challenges, the regimes were still able to make incremental updates to the control lists and to advance technical discussions. The inability of the WA to find consensus on several control list additions due to a Russian veto led a growing number of states and the EU to adopt controls on items that had been proposed and discussed within the WA.

EU controls
During 2025 the EU took steps to bolster its common legal framework for controls on the export, brokering, transit and trans-shipment of military items and dual-use items. The EU completed a review of the EU common position on arms exports and adopted amendments that strengthened the common position’s language on risk assessment and that aimed to facilitate the joint production of military equipment in the EU and arms transfers to Ukraine. In addition, the European Commission sought to amend the system that regulates intra-EU transfers of military equipment also in a bid to facilitate joint production. However, some of these proposals met resistance from EU member states that are concerned about losing national oversight of their export control policies.

Number of Arms Trade Treaty states parties submitting annual reports, 2019–24 (as extracted)
- % of due reports submitted (2019–24): 77%, 77%, 73%, 73%, 72%, 72% (data as extracted)
- Required to submit a report / Report submitted (2019–24): numbers shown in original extraction (chart).

Annexes

Arms control and disarmament agreements in force, 1 January 2026
- 1925 — Protocol for the Prohibition of the Use in War of Asphyxiating, Poisonous or Other Gases, and of Bacteriological Methods of Warfare (1925 Geneva Protocol)
- 1948 — Convention on the Prevention and Punishment of the Crime of Genocide (Genocide Convention)
- 1949 — Geneva Convention (IV) Relative to the Protection of Civilian Persons in Time of War; and 1977 Protocols I and II Relating to the Protection of Victims of International and Non‑International Armed Conflicts
- 1959 — Antarctic Treaty
- 1963 — Treaty Banning Nuclear Weapon Tests in the Atmosphere, in Outer Space and Under Water (Partial Test-Ban Treaty, PTBT)
- 1967 — Treaty on Principles Governing the Activities of States in the Exploration and Use of Outer Space, Including the Moon and Other Celestial Bodies (Outer Space Treaty)
- 1967 — Treaty for the Prohibition of Nuclear Weapons in Latin America and the Caribbean (Treaty of Tlatelolco)
- 1968 — Treaty on the Non-Proliferation of Nuclear Weapons (Non-Proliferation Treaty, NPT)
- 1971 — Treaty on the Prohibition of the Emplacement of Nuclear Weapons and other Weapons of Mass Destruction on the Seabed and the Ocean Floor and in the Subsoil thereof (Seabed Treaty)
- 1972 — Convention on the Prohibition of the Development, Production and Stockpiling of Bacteriological (Biological) and Toxin Weapons and on their Destruction (Biological and Toxin Weapons Convention, BWC)
- 1974 — Treaty on the Limitation of Underground Nuclear Weapon Tests (Threshold Test-Ban Treaty, TTBT)
- 1976 — Treaty on Underground Nuclear Explosions for Peaceful Purposes (Peaceful Nuclear Explosions Treaty, PNET)
- 1977 — Convention on the Prohibition of Military or Any Other Hostile Use of Environmental Modification Techniques (Enmod Convention)
- 1980 — Convention on the Physical Protection of Nuclear Material and Nuclear Facilities
- 1981 — Convention on Prohibitions or Restrictions on the Use of Certain Conventional Weapons which may be Deemed to be Excessively Injurious or to have Indiscriminate Effects (CCW Convention, or ‘Inhumane Weapons’ Convention)
- 1985 — South Pacific Nuclear Free Zone Treaty (Treaty of Rarotonga)
- 1990 — Treaty on Conventional Armed Forces in Europe (CFE Treaty)
- 1992 — Treaty on Open Skies
- 1993 — Convention on the Prohibition of the Development, Production, Stockpiling and Use of Chemical Weapons and on their Destruction (Chemical Weapons Convention, CWC)
- 1995 — Treaty on the Southeast Asia Nuclear Weapon-Free Zone (Treaty of Bangkok)
- 1996 — African Nuclear-Weapon-Free Zone Treaty (Treaty of Pelindaba)
- 1996 — Agreement on Sub-Regional Arms Control (Florence Agreement)
- 1997 — Inter-American Convention Against the Illicit Manufacturing of and Trafficking in Firearms, Ammunition, Explosives, and Other Related Materials (CIFTA)
- 1997 — Convention on the Prohibition of the Use, Stockpiling, Production and Transfer of Anti-Personnel Mines and on their Destruction (APM Convention)
- 1999 — Inter-American Convention on Transparency in Conventional Weapons Acquisitions
- 2001 — Protocol on the Control of Firearms, Ammunition and other related Materials in the Southern African Development Community (SADC) Region
- 2004 — Nairobi Protocol for the Prevention, Control and Reduction of Small Arms and Light Weapons in the Great Lakes Region and the Horn of Africa
- 2006 — ECOWAS Convention on Small Arms and Light Weapons, their Ammunition and Other Related Materials
- 2006 — Treaty on a Nuclear-Weapon-Free Zone in Central Asia (Treaty of Semipalatinsk)
- 2008 — Convention on Cluster Munitions
- 2010 — Treaty on Measures for the Further Reduction and Limitation of Strategic Offensive Arms (New START)
- 2010 — Central African Convention for the Control of Small Arms and Light Weapons, Their Ammunition and All Parts and Components That Can Be Used for Their Manufacture, Repair and Assembly (Kinshasa Convention)
- 2011 — Vienna Document 2011 on Confidence- and Security-Building Measures
- 2013 — Arms Trade Treaty (ATT)
- 2017 — Treaty on the Prohibition of Nuclear Weapons (TPNW)

Agreements not yet in force, 1 January 2026
- 1996 — Comprehensive Nuclear-Test-Ban Treaty (CTBT)
- 1999 — Agreement on Adaptation of the CFE Treaty

Security cooperation bodies — developments in 2025 included:
- The Maldives joined the International Atomic Energy Agency;
- Indonesia and Saudi Arabia joined the BRICS intergovernmental organization;
- Armenia, Syria and Uzbekistan joined the Global Coalition against Daesh;
- A record seven member countries remained suspended from participation in the African Union as a result of military coups;
- Timor-Leste joined the Association of Southeast Asian Nations;
- Burkina Faso, Mali and Niger withdrew from the Economic Community of West African States;
- Nigeria joined the Proliferation Security Initiative.

Chronology 2025, selected events
- 20 Jan. — The USA initiates withdrawal from the World Health Organization.
- 18 Feb. — Russia and the USA agree to normalize certain diplomatic relations.
- 18 Mar. — Israel launches widespread attacks in Gaza, ending a ceasefire agreed in Jan.
- 11–14 Apr. — More than 1500 civilians are killed in attacks on Sudan’s largest camp for internally displaced people.
- 7–10 May — India and Pakistan exchange intense cross-border fire.
- 13–24 June — Iran–Israel 12-day war includes US attacks on 3 Iranian nuclear facilities on 22 June.
- 24–28 July — Armed conflict between Cambodia and Thailand is ended by a ceasefire on 28 July.
- 22 Aug. — Famine is confirmed in Gaza City for the first time.
- 2 Sep. — The USA carries out the first of a series of lethal strikes on small civilian boats in or near Venezuelan waters.
- 10 Oct. — A ceasefire is agreed in the Gaza war as part of a USA-backed peace plan, but numerous attacks continue.
- 22–23 Nov. — The first Group of 20 (G20) intergovernmental summit in Africa emphasizes African priorities, including a new critical minerals framework.
- 1 Dec. — The first formal UN text is agreed on the danger of artificial intelligence in nuclear command, control and communications systems.

SIPRI Databases
- SIPRI Military Expenditure Database: Gives the annual military spending of countries since 1949, allowing comparison of countries’ military spending in local currency at current prices; in US dollars at current prices; in US dollars at constant prices and exchange rates; and as a share of gross domestic product.
- SIPRI Arms Industry Database: Contains annual data on total revenues and revenues from arms sales and military services since 2002 for the 100 companies with the highest arms revenues in the world. Data for Chinese companies is included for the years from 2015 onwards.
- SIPRI Arms Transfers Database: Shows all international transfers of major conventional arms since 1950. It is the most comprehensive publicly available source of information on international arms transfers.
- SIPRI Arms Embargoes Database: Gives information on all arms embargoes that have been implemented by an international organization, such as the European Union or the United Nations, or by a group of nations. All embargoes that are in force, or have been in force since 1998, are included.
- SIPRI National Reports Database: Provides links to all publicly accessible national reports on arms exports and is constantly updated to include links to newly published national reports on arms exports.
- SIPRI Multilateral Peace Operations Database: Offers information on all UN and non-UN peace operations conducted since 2000, including location, dates of deployment and operation, mandate, participating countries, number of personnel, budgets and fatalities.

The SIPRI databases can be accessed from the SIPRI website.

How to order SIPRI Yearbook 2026
SIPRI Yearbook 2026: Armaments, Disarmament and International Security  
Published in print and online by Oxford University Press  
ISBN 978–0–19–792320–7  
More information is available at www.sipriyearbook.org

Signalistgatan 9  
SE-169 72 Solna, Sweden  
Telephone: +46 8 655 97 00  
Email: sipri@sipri.org  
Internet: www.sipri.org

Knowledge for a peaceful and secure world — 60 years.

## Live News Feed Collection and Storage
Currently only from MoneyControl

In [4]:
import requests
from bs4 import BeautifulSoup

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

In [5]:
# ------------------------------------------------------------------------
# Prepares a list of MoneyControl WORLD URLs to be scraped
# ------------------------------------------------------------------------

def get_mc_links():

    url = "https://www.moneycontrol.com/world/"

    res = requests.get(url, headers=HEADERS)
    soup = BeautifulSoup(res.text, "html.parser")

    links = set()

    for a in soup.find_all("a", href=True):

        href = a["href"]

        if (
            href.startswith("https://www.moneycontrol.com/")
            and "/world/" in href
        ):
            links.add(href)

    print(f"WORLD links located : {len(links)}")

    return list(links)

In [6]:
mc_links = get_mc_links()

WORLD links located : 60


In [7]:
# ------------------------------------------------------------------------
# Extracts text from ONE MoneyControl URL
# ------------------------------------------------------------------------

def extract_mc(url):

    try:
        res = requests.get(url, headers=HEADERS)
        soup = BeautifulSoup(res.text, "html.parser")

        # Title
        h1 = soup.find("h1")

        if not h1:
            return None

        title = h1.get_text(strip=True)

        # Target article content container
        container = (
            soup.find("div", class_="content_wrapper") or
            soup.find("div", id="contentdata")
        )

        if not container:
            return None

        paragraphs = container.find_all("p")

        clean = []

        for p in paragraphs:

            text = p.get_text(" ", strip=True)

            # Remove junk
            if any(x in text.lower() for x in [
                "follow us",
                "invest now",
                "advertisement",
                "click here",
                "watch video"
            ]):
                continue

            if len(text.split()) < 8:
                continue

            clean.append(text)

        if len(clean) < 3:
            return None

        return {
            "title": title,
            "content": " ".join(clean[:6]),
            "source": "Moneycontrol",
            "url": url
        }

    except Exception as e:
        print("Error:", e)
        return None

In [8]:
# ------------------------------------------------------------------------
# Extracts valid stories from discovered MoneyControl links
# ------------------------------------------------------------------------

def get_stories(mc_links, batch_size=10, delay=0.5):

    stories = []
    total = len(mc_links)

    for i, u in enumerate(mc_links, 1):

        art = extract_mc(u)

        if art:
            stories.append(art)

        # Status + throttle after every batch of 10
        if i % batch_size == 0:
            print(
                f"Processed {i}/{total} | Valid stories: {len(stories)}"
            )
            time.sleep(delay)

    # Print final partial batch, if any
    if total % batch_size != 0:
        print(
            f"Processed {total}/{total} | Valid stories: {len(stories)}"
        )

    print("-" * 40)
    print(f"Links examined : {total}")
    print(f"Valid stories  : {len(stories)}")

    return stories

In [11]:
stories = get_stories(mc_links)

Processed 10/60 | Valid stories: 9
Processed 20/60 | Valid stories: 12
Processed 30/60 | Valid stories: 19
Processed 40/60 | Valid stories: 25
Processed 50/60 | Valid stories: 32
Processed 60/60 | Valid stories: 41
----------------------------------------
Links examined : 60
Valid stories  : 41


In [12]:
# ------------------------------------------------------------------------
# Score stories for Middle East Oil Security relevance
# ------------------------------------------------------------------------

def score_MOS(article):

    text = (
        article["title"]
        + " "
        + article["content"]
    ).lower()

    score = 0

    high = [
        "strait of hormuz",
        "hormuz",
        "iran",
        "iranian",
        "crude oil",
        "oil price",
        "brent",
        "middle east",
        "persian gulf"
    ]

    geopolitical = [
        "united states",
        "u.s.",
        "israel",
        "missile",
        "drone",
        "air strike",
        "airstrike",
        "attack",
        "war",
        "military",
        "sanctions",
        "retaliation",
        "ceasefire"
    ]

    oil_security = [
        "tanker",
        "shipping",
        "oil supply",
        "oil exports",
        "oil production",
        "refinery",
        "opec",
        "opec+",
        "saudi arabia",
        "uae",
        "qatar",
        "gulf"
    ]

    india = [
        "india",
        "indian oil",
        "ioc",
        "bharat petroleum",
        "hindustan petroleum",
        "reliance industries",
        "oil import",
        "petroleum ministry"
    ]

    for k in high:
        if k in text:
            score += 4

    for k in geopolitical:
        if k in text:
            score += 2

    for k in oil_security:
        if k in text:
            score += 2

    for k in india:
        if k in text:
            score += 2

    return score

In [13]:
# ------------------------------------------------------------------------
# Score and sort stories by MOS relevance
# ------------------------------------------------------------------------

for s in stories:
    s["score"] = score_MOS(s)

stories = sorted(
    stories,
    key=lambda x: x["score"],
    reverse=True
)

for i, s in enumerate(stories, 1):
    print(f"{i:3}. [{s['score']:2}] {s['title']}")

  1. [26] Iran says US response on Hormuz plan expected 'hopefully' tomorrow after Qatar-mediated talks
  2. [24] ‘It will not be long’: Iran’s Khamenei warns US will be forced out of Arabian Sea after ‘painful blows’
  3. [24] Tehran awaits Trump’s response on war-end plan, says Araghchi after US-Iran talks via Qatar in New York
  4. [24] US, Iran set for separate talks with mediators over Strait of Hormuz
  5. [22] US should not dare approach Strait of Hormuz: Mojtaba Khamenei says Iran ready to ‘humiliate’ enemies
  6. [20] Iran agrees to halt uranium enrichment in exchange for sanctions relief, frozen assets: Report
  7. [20] USS George Washington in Arabian Sea after Khamenei’s Hormuz warning? CENTCOM shares video
  8. [18] ‘Hoax’: Trump denies offering Iran sanctions relief as Tehran rules out nuclear concessions
  9. [16] ‘Iran’s rial hits record lows’: US Treasury Secretary Bessent says sanctions will continue to cripple Tehran’s nuclear funding
 10. [12] Europe's LNG shift is 

In [14]:
# ------------------------------------------------------------------------
# Inspect top MOS-ranked stories
# ------------------------------------------------------------------------

for i, s in enumerate(stories[:15], 1):

    print("=" * 80)
    print(f"{i}. SCORE: {s['score']}")
    print(s["title"])
    print()
    print(s["content"])
    print()

1. SCORE: 26
Iran says US response on Hormuz plan expected 'hopefully' tomorrow after Qatar-mediated talks

Iranian Foreign Minister Abbas Araghchi said Tehran held fresh indirect discussions with the United States through Qatari mediators in New York, with Washington's response to Iran's proposal for reopening the Strait of Hormuz expected as early as Wednesday. Araghchi said the latest talks focused on ways to address conditions put forward by Tehran for resolving the crisis and restoring shipping through the strategic waterway. The Iranian foreign minister said Qatari mediators had presented new ideas aimed at narrowing differences between Tehran and Washington. The proposals are now being conveyed to the US side, after which Washington's response will be passed back to Iran. “I hope this will happen by tomorrow," Araghchi said, referring to the expected US response. Araghchi said he would leave New York for Tehran within hours, while the Qatari mediators would communicate the US po

In [ ]:
#Check balance
# https://platform.openai.com/usage?utm_source=chatgpt.com